# Malpasset dam break (viscous) — new SystemModel + symbolic-Riemann pipeline

Port of `tutorials/firedrake/malpasset_viscous.py` to the new
`FiredrakeHyperbolicSolver` (SystemModel → symbolic Riemann → UFL
runtime).  Same physical setup and geometry as the original.

The original uses a hand-rolled SWE class with `[b, h, hu, hv]`
state, `[hinv]` aux, and inserts a hand-written DG(0) TPFA viscous
block into the weak form.  Here we:

- define an equivalent `MalpassetSWE` Model that returns
  `flux`, `nonconservative_matrix`, `source` and `diffusion_matrix`
  in the new operator-form convention;
- let `SystemModel.from_model` extract the operators;
- let `Rusanov(SystemModel).to_runtime_ufl()` lower the Riemann
  numerics to UFL;
- rely on the solver's built-in TPFA (DG(0)) and IP-DG (DG(1+))
  diffusion paths — no per-app weak-form override.

Two runs at the bottom: DG(0), then DG(1) with vertex-based limiter.

In [ ]:
import os
import time

import numpy as np
import sympy as sp
from sympy import Matrix, sqrt, Piecewise

import firedrake as fd
import meshio

from zoomy_core.fvm.solver_numpy import Settings
from zoomy_core.fvm.riemann_solvers import Rusanov
from zoomy_core.misc.misc import Zstruct, ZArray
from zoomy_core.model.basemodel import Model
import zoomy_core.model.boundary_conditions as BC
import zoomy_core.misc.misc as misc

from zoomy_firedrake.firedrake_solver import FiredrakeHyperbolicSolver

## Physical parameters and inputs

In [ ]:
MANNING_N = float(os.environ.get("MALPASSET_MANNING", "0.033"))
EPS_WD = float(os.environ.get("MALPASSET_EPS_WD", "1e-2"))
H_FRICTION_FLOOR = float(os.environ.get("MALPASSET_H_FRICTION", "0.5"))
NU = float(os.environ.get("MALPASSET_NU", "1.0"))
TIME_END = float(os.environ.get("MALPASSET_TIME_END", "5.0"))
CFL = float(os.environ.get("MALPASSET_CFL", "0.5"))

main_dir = misc.get_main_directory()
INPUT_MESH = os.path.join(main_dir, "data", "malpasset",
                          "geo_malpasset-small.msh")
assert os.path.exists(INPUT_MESH), f"missing mesh: {INPUT_MESH}"

# Loaded once and reused for IC projection.
MESHIO_MESH = meshio.read(INPUT_MESH)

## SWE Model — 4-component state, aux ``hinv``, depth-weighted
viscosity

Matches the original `malpasset_viscous.py` operator-by-operator,
but in the new ``Model → SystemModel`` convention.

In [ ]:
class MalpassetSWE(Model):
    """SWE with `[b, h, hu, hv]` state and `[hinv]` aux.

    - ``flux``: convective only (``hu⊗u``).  The hydrostatic pressure
      ``½ g h² I`` is moved into ``nonconservative_matrix`` so the
      well-balancing handshake with the bathymetry slope is preserved
      (Audusse-style).
    - ``nonconservative_matrix``: ``g h ∂_d b`` and ``g h ∂_d h`` on the
      momentum rows.
    - ``source``: Manning bed friction with floored ``h^{-1/3}``.
    - ``diffusion_matrix``: ``A[hu_i, hu_i, d, d] = ν · h``.
    - ``eigenvalues``: switched off where ``h ≤ eps`` (dry cells get
      zero wave speed).
    """

    def __init__(self, *, g=9.81, n=MANNING_N, nu=NU, eps=EPS_WD, **kw):
        super().__init__(
            dimension=2,
            variables=["b", "h", "hu", "hv"],
            aux_variables=["hinv"],
            parameters={
                "g":  (float(g),  "positive"),
                "n":  (float(n),  "non-negative"),
                "nu": (float(nu), "non-negative"),
                "eps": (float(eps), "positive"),
            },
            eigenvalue_mode="symbolic",
            **kw,
        )

    # -- Convenience -----------------------------------------------------
    def _primitives(self):
        v = self.variables
        a = self.aux_variables
        return v.b, v.h, v.hu, v.hv, a.hinv

    # -- Operators -------------------------------------------------------
    def flux(self):
        _, h, hu, hv, hinv = self._primitives()
        F = Matrix.zeros(4, 2)
        # mass equation
        F[1, 0] = hu
        F[1, 1] = hv
        # momentum: pure convective (no pressure here — see NCP)
        F[2, 0] = hu * hu * hinv
        F[2, 1] = hu * hv * hinv
        F[3, 0] = hu * hv * hinv
        F[3, 1] = hv * hv * hinv
        return ZArray(F)

    def nonconservative_matrix(self):
        _, h, _, _, _ = self._primitives()
        g = self._parameter_symbols.g
        N = ZArray.zeros(4, 4, 2)
        # Momentum_x: depends on ∂_x b and ∂_x h
        N[2, 0, 0] = g * h
        N[2, 1, 0] = g * h
        # Momentum_y: depends on ∂_y b and ∂_y h
        N[3, 0, 1] = g * h
        N[3, 1, 1] = g * h
        return N

    def source(self):
        _, h, hu, hv, hinv = self._primitives()
        p = self._parameter_symbols
        u = hu * hinv
        w = hv * hinv
        u_mag = sqrt(u * u + w * w + 1e-12)
        # Manning friction with bounded ``h^(-1/3)``: floor the depth at
        # ``H_FRICTION_FLOOR`` so friction stays finite at the wet/dry
        # interface.  Wet/dry "deactivation" happens naturally —
        # ``hu = hv = 0`` ⇒ ``u_mag = 0`` ⇒ friction = 0 — and at very
        # low depths ``hinv`` (from ``update_aux_variables``) is also
        # already floored.  We deliberately avoid ``Piecewise`` /
        # ``conditional`` here because the SystemModel auto-derives the
        # source Jacobian via ``sp.diff`` and conditionals are not
        # differentiable in SymPy.
        h_safe = sp.Max(h, sp.Float(H_FRICTION_FLOOR))
        friction_div = h_safe ** (-sp.Rational(1, 3))
        factor = -p.n ** 2 * p.g * friction_div * u_mag
        S_b = sp.S.Zero
        S_h = sp.S.Zero
        S_hu = factor * hu
        S_hv = factor * hv
        return ZArray([S_b, S_h, S_hu, S_hv])

    def diffusion_matrix(self):
        _, h, _, _, _ = self._primitives()
        nu = self._parameter_symbols.nu
        A = sp.MutableDenseNDimArray.zeros(4, 4, 2, 2)
        for i_row in (2, 3):       # hu, hv
            for d in (0, 1):
                A[i_row, i_row, d, d] = nu * h
        return ZArray(A)

    def eigenvalues(self):
        _, h, hu, hv, hinv = self._primitives()
        p = self._parameter_symbols
        n = self.normal
        u = hu * hinv
        w = hv * hinv
        un = u * n.n0 + w * n.n1
        # Floor depth in the spectral radius so wet/dry doesn't blow it
        # up; sqrt(g·max(h, eps)) stays finite and tends to √(g·eps) on
        # dry cells.
        c = sqrt(p.g * sp.Max(h, p.eps))
        return ZArray([sp.S.Zero, un, un - c, un + c])

    def update_aux_variables(self):
        """hinv = 1 / max(h, eps)."""
        v = self.variables
        p = self._parameter_symbols
        h_safe = sp.Max(v.h, p.eps)
        return ZArray([1 / h_safe])

## Solver subclass: meshio-driven initial condition

Only override `set_initial_condition` to load `B, H, HU, HV` from the
point data of the meshio file.  Everything else (Riemann solver,
weak forms, diffusion path) comes from the base solver.

In [ ]:
def _build_vertex_permutation(fd_mesh, meshio_mesh, decimal=12):
    """See `malpasset_baseline.py` — Firedrake reorders nodes."""
    dim = fd_mesh.geometric_dimension()
    coords_fd = np.round(fd_mesh.coordinates.dat.data_ro[:, :dim], decimal)
    coords_mio = np.round(meshio_mesh.points[:, :dim], decimal)
    lookup = {tuple(c): i for i, c in enumerate(coords_mio)}
    perm = np.empty(coords_fd.shape[0], dtype=np.int64)
    for j, c in enumerate(coords_fd):
        perm[j] = lookup[tuple(c)]
    return perm


class MalpassetSolver(FiredrakeHyperbolicSolver):
    """FiredrakeHyperbolicSolver with the Malpasset IC loader."""

    def set_initial_condition(self, Q, model):
        mesh = Q.function_space().mesh()
        # CG1 staging space so we can write the raw vertex point data
        # before projecting onto the (possibly DG1) target space.
        V_CG = fd.VectorFunctionSpace(mesh, "CG", 1,
                                      dim=Q.function_space().value_size)
        Q_CG = fd.Function(V_CG)
        perm = _build_vertex_permutation(mesh, MESHIO_MESH)
        pd = MESHIO_MESH.point_data
        Q_CG.dat.data[:, 0] = pd["B"][perm]
        Q_CG.dat.data[:, 1] = pd["H"][perm]
        Q_CG.dat.data[:, 2] = (pd["H"] * pd["U"])[perm]
        Q_CG.dat.data[:, 3] = (pd["H"] * pd["V"])[perm]
        Q.project(Q_CG)

## Boundary conditions and Settings

Same tag list as the original; the Malpasset mesh has no named
Physical Curve groups, so the kernel matches no facets — boundaries
behave as free / extrapolating.

In [ ]:
bcs = BC.BoundaryConditions(
    [BC.Wall(tag="wall"),
     BC.Wall(tag="inflow"),
     BC.Wall(tag="outflow")]
)

settings = Settings(
    name="malpasset-viscous-v2",
    output=Zstruct(
        directory="outputs/firedrake_viscous_v2",
        snapshots=10, filename="dg", clean_directory=True,
    ),
)

## Run — DG(0) with TPFA viscous flux

In [ ]:
def run(dg_degree=0, limiter="none", time_end=TIME_END, tag=""):
    model = MalpassetSWE()
    out_tag = tag or f"dg{dg_degree}_lim{limiter}"
    s = Settings(
        name=f"malpasset-{out_tag}",
        output=Zstruct(directory=f"outputs/firedrake_viscous_v2_{out_tag}",
                       snapshots=10, filename="dg", clean_directory=True),
    )
    solver = MalpassetSolver(
        settings=s,
        time_end=time_end,
        CFL=CFL,
        dg_degree=dg_degree,
        limiter=limiter,
        riemann_solver_cls=Rusanov,
    )
    t0 = time.perf_counter()
    solver.solve(INPUT_MESH, model)
    t1 = time.perf_counter()
    print(f"[malpasset {out_tag}] wall_time={t1 - t0:.2f}s")
    return solver

In [ ]:
if __name__ == "__main__":
    print(f"[malpasset] ν={NU}  time_end={TIME_END}  CFL={CFL}")
    solver_dg0 = run(dg_degree=0, limiter="none", tag="dg0_tpfa")
    solver_dg1 = run(dg_degree=1, limiter="vertex", tag="dg1_ipdg_vert")